<a href="https://colab.research.google.com/github/FabioFloris02/NLP2026_Floris_Sonzini_Parenti_Sarra_Rossi/blob/rag/RAG_Ealuation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
JSON_PATH = "/content/drive/MyDrive/Progetto-NLP/Branch-rag/log.jsonl"

In [ ]:
import json
import re

N = 10 # Il numero delle ultime domande della competizione

# 1. RIPARATORE DI ARRAY CONCATENATI
with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw_content = f.read()

# Trova tutte le parentesi chiuse seguite da parentesi aperte (anche con spazi/a capo in mezzo)
# e le sostituisce con una virgola.
# Es: [{...}] \n [{...}]  DIVENTA  [{...} , {...}]
fixed_content = re.sub(r'\]\s*\[', ',', raw_content)

# Ora possiamo leggerlo tranquillamente come un'unica lista piatta!
log = json.loads(fixed_content)

# 2. Filtraggio degli errori (Ignorando i Timeout)
errori_totali = [entry for entry in log if entry.get("correct") is False and entry.get("timed_out") is False]

# 3. Estrazione degli ultimi N errori
ultimi_errori = errori_totali[-N:] if N > 0 else errori_totali

print(f"File riparato! Il log contiene {len(log)} domande totali.")
print(f"Di cui {len(errori_totali)} sono errori.")
print(f"Selezionati gli ultimi {len(ultimi_errori)} per la valutazione.\n")

# 4. Funzione per estrarre il contesto pulito
def estrai_contesto(prompt_completo):
    match = re.search(r"<context>(.*?)</context>", prompt_completo, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return "Contesto non trovato."

wrong_answers = []
# 5. Pipeline per preparare i dati per il Giudice
for i, entry in enumerate(ultimi_errori):
    domanda = entry["question"]
    opzioni = "\n".join(entry["options"])
    risposta_sbagliata = entry["chosen_option"]
    contesto_pulito = estrai_contesto(entry["complete_prompt"])

    errore_dal_log = {
        "query": domanda,
        "options": opzioni,
        "context": contesto_pulito,
        "wrong_answer": risposta_sbagliata
    }

    wrong_answers.append(errore_dal_log)

    print(f"================ ERROR {i+1}/{len(ultimi_errori)} ================")
    print(f"Q: {domanda}")
    print(f"Il modello ha risposto in modo errato: {risposta_sbagliata}")
    print(f"Context recuperato in quel momento (primi 100 char): {contesto_pulito[:100]}...\n")

    # ---------------------------------------------------------
    # Qui è dove chiami la tua API Groq/Llama 70B
    # sentenza = judge_rag_error(...)
    # print(f"GIUDIZIO: {sentenza['error_type']} - {sentenza['explanation']}")
    # ---------------------------------------------------------

File riparato! Il log contiene 271 domande totali.
Di cui 42 sono errori.
Selezionati gli ultimi 10 per la valutazione.

================ ERROR 1/10 ================
Q: What was the primary factor that allowed the Neo-Assyrian Empire to become the largest empire in history up to that point?
Il modello ha risposto in modo errato: The development of a sophisticated state communication system
Context recuperato in quel momento (primi 100 char): L'Impero neo-assiro emerse nel X secolo a.C. e raggiunse il suo apice nell'VIII e VII secolo a.C., s...

================ ERROR 2/10 ================
Q: Which of the following best describes Marilyn Monroe's relationship with 20th Century-Fox during her career?
Il modello ha risposto in modo errato: She had a harmonious and mutually beneficial relationship with the studio throughout her career.
Context recuperato in quel momento (primi 100 char): Una biografia piuttosto libera di Marilyn Monroe, arricchita da una serie di reali aneddoti sulla su...

In [ ]:
!pip install mistralai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.38.0
    Uninstalling opentelemetry-api-1.38.0:
      Successfully uninstalled opentelemetry-api-1.38.0
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.59b0
    Uninstalling opentelemetry-semantic-conventions-0.59b0:
      Successfully uninstalled opentelemetry-semantic-conventions-0.59b0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-ap

In [ ]:
from mistralai.client import Mistral
import time
from google.colab import userdata

#Add API key in colab secrets
API_KEY=userdata.get('mistral')
client = Mistral(api_key=API_KEY)

In [ ]:
import json
import time
import re

# Funzione helper per tagliare il contesto fuori dal prompt lungo
def estrai_contesto(prompt_completo):
    match = re.search(r"<context>(.*?)</context>", prompt_completo, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return "Contesto non trovato."

def judge_rag_error(domanda, opzioni, contesto, risposta_sbagliata):
    system_prompt = """Sei un Giudice Supremo per la valutazione di sistemi RAG (Retrieval-Augmented Generation).
Ti verrà fornita una Domanda a scelta multipla, le Opzioni, il Contesto letto dal sistema e la Risposta (sbagliata) data dal sistema.

Devi classificare il motivo dell'errore in UNA di queste 4 categorie:
A: RETRIEVAL_FAILURE -> Il contesto non contiene le informazioni necessarie per rispondere.
B: REASONING_FAILURE -> Il contesto contiene la risposta chiara e un'opzione combacia, ma il sistema ha sbagliato a sceglierla.
C: DATASET_MISMATCH -> Il contesto contiene un'informazione, ma non combacia con nessuna delle opzioni proposte.
D: HALLUCINATION -> Il contesto non conteneva la risposta, ma il sistema ha inventato/indovinato un'opzione invece di arrendersi.

Restituisci ESCLUSIVAMENTE un oggetto JSON valido con questo formato:
{"error_type": "A", "explanation": "Breve spiegazione del perché hai scelto questa categoria."}"""

    user_prompt = f"""
[DOMANDA]: {domanda}
[OPZIONI]: {opzioni}
[CONTESTO LETTO DAL SISTEMA]: {contesto}
[RISPOSTA SBAGLIATA DEL SISTEMA]: {risposta_sbagliata}

Giudizio:"""

    max_attempts = 5
    for attempt in range(max_attempts): # Aggiunto il ciclo for per i Retry!
        try:
            response = client.chat.complete(
                model="mistral-large-latest",
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=0.0
            )

            result_text = response.choices[0].message.content

            # 🔴 PULIZIA DEL MARKDOWN: Togliamo ```json e ```
            result_text = result_text.replace("```json", "").replace("```", "").strip()

            # Trasformiamo la stringa pulita in un vero dizionario Python
            result_json = json.loads(result_text)

            time.sleep(2) # Pausa di rispetto per le API
            return result_json # Restituiamo il dizionario!

        except Exception as e:
            if "rate limit" in str(e).lower() or "429" in str(e):
                wait_time = 4 ** attempt
                print(f"Rate limited. Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                # Se c'è un errore strano (es. JSON corrotto), restituisce un log di errore ma non fa crashare tutto
                print(f"Errore inatteso: {e}")
                return {"error_type": "UNKNOWN", "explanation": str(e)}

    return {"error_type": "UNKNOWN", "explanation": "Max retries reached"}

In [ ]:
# Inizializziamo il dizionario delle statistiche
statistiche = {
    "A": 0, # Retrieval Failure
    "B": 0, # Reasoning Failure
    "C": 0, # Dataset Mismatch
    "D": 0, # Hallucination
    "UNKNOWN": 0 # Errori API
}

#print(f"🚀 Inizio valutazione di {len(wrong_answers)} errori...\n")

for indice, i in enumerate(wrong_answers):
    print("-" * 50)
    print(f"Analisi domanda {indice + 1}/{len(wrong_answers)}: {i['query'][:50]}...")

    # 🔴 CORREZIONE: Peschiamo i dati dinamici dal dizionario 'i'
    domanda_corrente = i["query"]
    opzioni_correnti = i["options"]
    risposta_errata = i["wrong_answer"]
    contesto_corrente = i["context"] # Corrected: Access 'context' directly as it's already extracted

    # Chiamiamo la funzione del Giudice
    sentenza = judge_rag_error(
        domanda=domanda_corrente,
        opzioni=opzioni_correnti,
        contesto=contesto_corrente,
        risposta_sbagliata=risposta_errata
    )

    # Estraiamo i dati dal JSON (che ora è un vero dizionario)
    tipo_errore = sentenza.get("error_type", "UNKNOWN")
    spiegazione = sentenza.get("explanation", "Nessuna spiegazione")

    print(f"ESITO: Tipo {tipo_errore}")
    print(f"MOTIVO: {spiegazione}")

    # Aggiorniamo le statistiche
    if tipo_errore in statistiche:
        statistiche[tipo_errore] += 1
    else:
        statistiche["UNKNOWN"] += 1

# ==========================================
# STAMPA FINALE DELLE STATISTICHE
# ==========================================
print("\n" + "="*50)
print("📊 REPORT FINALE DELLE CAUSE DI ERRORE")
print("="*50)

totale_errori = len(wrong_answers)
for tipo, conteggio in statistiche.items():
    if totale_errori > 0:
        percentuale = (conteggio / totale_errori) * 100
        print(f"Errore di tipo {tipo}: {conteggio} su {totale_errori} ({percentuale:.1f}%)")

print("="*50)

--------------------------------------------------
Analisi domanda 1/10: What was the primary factor that allowed the Neo-A...
ESITO: Tipo B
MOTIVO: Il contesto fornito menziona esplicitamente che 'la capacità di incorporare efficientemente le terre conquistate nel suo sistema amministrativo' fu una delle innovazioni chiave dell'Impero neo-assiro, citando anche il sistema di comunicazione come un fattore importante ma non primario per la grandezza dell'impero. Tuttavia, il sistema ha scelto l'opzione relativa al sistema di comunicazione, che, pur essendo rilevante, non è indicata come il fattore primario nel contesto.
--------------------------------------------------
Analisi domanda 2/10: Which of the following best describes Marilyn Monr...
ESITO: Tipo B
MOTIVO: Il contesto fornito contiene informazioni chiare sul rapporto conflittuale di Marilyn Monroe con la 20th Century-Fox, menzionando il suo contratto iniziale ma anche le difficoltà e le tensioni (ad esempio, il passaggio a Colu